In [10]:
# BLOCK 1: features, files, settings etc.
import multiprocessing
import numpy as np
import pandas as pd
import joblib
from joblib import Parallel, delayed
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, RationalQuadratic, WhiteKernel
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

DATA_FILE = "Features_29_13494.xlsx"
BLIND_FILE = "blind_holdout_10_materials.xlsx"
target = "Temp"
n_jobs = multiprocessing.cpu_count()
TEST_SIZE = 0.40
RANDOM_STATE = 42
NN_COMPARISON_SEEDS = [1, 2, 3, 4, 5]
NN_BLIND_SEEDS = [42, 52, 62, 72, 82]

df_full = pd.read_excel(DATA_FILE)
full_features = [
    c for c in df_full.columns
    if c not in {"Material", target}
]
consensus_features = [
    # 6/6 models
    'VEC',
    'wtd_mean_density',
    'metallic_fraction',
    'carrier_reservoir_ratio',
    'heavy_element_fraction',
    'wtd_variance_ionenergies',

    # 5/6 models
    'oxygen_fraction',
    'alkaline_fraction',
    'covalent_fraction',
    'light_element_fraction',
    'wtd_variance_atomic_weight',
    'stoich_variance_fraction',

    # 4/6 models
     'wtd_mean_electron_affinity',
     'wtd_mean_thermal_conductivity',
     'cu_fraction'
]
strict_features = [
    'VEC',
    'wtd_mean_density',
    'metallic_fraction',
    'carrier_reservoir_ratio',
    'heavy_mass_fraction',
    'wtd_variance_ionenergies',
]

feature_sets = {"Pearson-29": full_features,"Consensus-15": consensus_features,"Strict-6": strict_features,}
EXPECTED_COUNTS = {"Pearson-29": 29, "Consensus-15": 15, "Strict-6": 6}
GPR_MODEL_FILES = {
    "Pearson-29": "GPR_RQ_Full-F29.pkl",
    "Consensus-15": "GPR_RQ_Consensus15.pkl",
    "Strict-6": "GPR_RQ_Strict6.pkl",
}

def validate_data(data, label):
    for name, features in feature_sets.items():
        if len(features) != EXPECTED_COUNTS[name] or len(set(features)) != len(features):
            raise ValueError(f"{name}: check feature count/duplicates in Block 1")
        missing = set(features + [target, "Material"]) - set(data.columns)
        if missing:
            raise ValueError(f"{label}: missing columns {sorted(missing)}")
        if not np.isfinite(data[features + [target]].to_numpy(dtype=float)).all():
            raise ValueError(f"{label}: non-finite feature/target values")

def load_blind_data():
    development = pd.read_excel(DATA_FILE, sheet_name="Sheet1")
    blind = pd.read_excel(BLIND_FILE, sheet_name="Blind_10_Materials")
    validate_data(development, "Development")
    validate_data(blind, "Blind")
    # Do not silently alter the original holdout or remove overlapping rows.
    overlap = set(development["Material"]) & set(blind["Material"])
    if overlap:
        raise ValueError(f"Development/blind material overlap: {sorted(map(str, overlap))}")
    print("Development:", development.shape, "Blind:", blind.shape)
    return development, blind

def metrics(y_true, prediction):
    return {"R2": r2_score(y_true, prediction),
            "RMSE": np.sqrt(mean_squared_error(y_true, prediction)),
            "MAE": mean_absolute_error(y_true, prediction)}

def save_blind(filename, performance, predictions):
    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        performance.to_excel(writer, sheet_name="Performance", index=False)
        predictions.to_excel(writer, sheet_name="Predictions", index=False)

def create_models():
    models = {}
    models['Linear'] = Pipeline([('scale', StandardScaler()), ('lr', LinearRegression())])
    models['Polynomial-D2'] = Pipeline([('scale', StandardScaler()), ('poly', PolynomialFeatures(degree=2, include_bias=False)), ('lr', LinearRegression())])
    models['SVR-RBF'] = Pipeline([('scale', StandardScaler()), ('svr', SVR(kernel='rbf', C=10, gamma='scale'))])
    models['Random Forest'] = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=n_jobs)
    models['XGBoost'] = XGBRegressor(n_estimators=500, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=n_jobs)
    models['LightGBM'] = LGBMRegressor(n_estimators=500, learning_rate=0.03, random_state=42, n_jobs=n_jobs, verbosity=-1)
    return models

def run_nn(seed, Xtr, ytr, Xte):
    nn = Pipeline([('scale', StandardScaler()), ('nn', MLPRegressor(hidden_layer_sizes=(128, 64, 32), max_iter=2000, random_state=seed))])
    nn.fit(Xtr, ytr)
    return nn.predict(Xte)

def nn_ensemble(Xtr, ytr, Xte, seeds, parallel=False):
    if parallel:
        predictions = Parallel(n_jobs=min(n_jobs, len(seeds)))(
            delayed(run_nn)(seed, Xtr, ytr, Xte) for seed in seeds)
    else:
        predictions = [run_nn(seed, Xtr, ytr, Xte) for seed in seeds]
    return np.mean(predictions, axis=0), np.std(predictions, axis=0)

def create_gpr_rbf():
    kernel = ConstantKernel(1.0) * RBF(length_scale=1.0, length_scale_bounds=(1e-05, 100000.0)) + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-05, 1))
    model = Pipeline([('scale', StandardScaler()), ('gpr', GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=42, n_restarts_optimizer=2))])
    return model

def create_gpr_rq():
    kernel = ConstantKernel(1.0) * RationalQuadratic(length_scale=1.0, alpha=1.0) + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-05, 1))
    model = Pipeline([('scale', StandardScaler()), ('gpr', GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=42, n_restarts_optimizer=2))])
    return model

def create_blind_gpr_rq():
    kernel = ConstantKernel(1.0) * RationalQuadratic(length_scale=1.0, alpha=1.0) + WhiteKernel(noise_level=0.1)
    return Pipeline([('scaler', StandardScaler()), ('gpr', GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=42, n_restarts_optimizer=2))])

validate_data(df_full, "Comparison database")
X_train_all, X_test_all, y_train, y_test = train_test_split(
    df_full[full_features], df_full[target], test_size=TEST_SIZE, random_state=RANDOM_STATE)
print("Feature counts:", {name: len(features) for name, features in feature_sets.items()})
print("Training:", X_train_all.shape, "Testing:", X_test_all.shape)

Feature counts: {'Pearson-29': 29, 'Consensus-15': 15, 'Strict-6': 6}
Training: (8096, 29) Testing: (5398, 29)


In [11]:
# BLOCK 2 — all non-GPR models: F29, F15, F6
results = []
for feature_name, features in feature_sets.items():
    Xtr, Xte = X_train_all[features], X_test_all[features]
    for model_name, model in create_models().items():
        print(feature_name, model_name)
        model.fit(Xtr, y_train)
        results.append({"Feature Set": feature_name, "Features": len(features),
                        "Model": "Linear Regression" if model_name == "Linear" else model_name,
                        **metrics(y_test, model.predict(Xte))})
    prediction, _ = nn_ensemble(Xtr, y_train, Xte, NN_COMPARISON_SEEDS, parallel=True)
    results.append({"Feature Set": feature_name, "Features": len(features),
                    "Model": "Deep Ensemble NN", **metrics(y_test, prediction)})

results_df = pd.DataFrame(results)
comparison_table = results_df.pivot_table(
    index="Model", columns="Feature Set", values=["R2", "RMSE", "MAE"])
comparison_table = comparison_table.reindex(
    columns=[(metric, name) for metric in ["R2", "RMSE", "MAE"] for name in feature_sets])
comparison_table.columns = [f"{metric}_{name}" for metric, name in comparison_table.columns]
model_order = ["LightGBM", "XGBoost", "Deep Ensemble NN", "Random Forest",
               "SVR-RBF", "Polynomial-D2", "Linear Regression"]
comparison_table = comparison_table.reindex(model_order).reset_index()
display(comparison_table)

Pearson-29 Linear
Pearson-29 Polynomial-D2
Pearson-29 SVR-RBF
Pearson-29 Random Forest
Pearson-29 XGBoost
Pearson-29 LightGBM
Consensus-15 Linear
Consensus-15 Polynomial-D2
Consensus-15 SVR-RBF
Consensus-15 Random Forest
Consensus-15 XGBoost
Consensus-15 LightGBM
Strict-6 Linear
Strict-6 Polynomial-D2
Strict-6 SVR-RBF
Strict-6 Random Forest
Strict-6 XGBoost
Strict-6 LightGBM


,Model,R2_Pearson-29,R2_Consensus-15,R2_Strict-6,RMSE_Pearson-29,RMSE_Consensus-15,RMSE_Strict-6,MAE_Pearson-29,MAE_Consensus-15,MAE_Strict-6
0,LightGBM,0.977135,0.974130,0.961924,4.921271,5.234660,6.350685,2.227938,2.509678,3.169841
1,XGBoost,0.976545,0.972730,0.960619,4.984425,5.374478,6.458545,2.222996,2.514730,3.231697
2,Deep Ensemble NN,0.973767,0.971316,0.946536,5.271257,5.512039,7.525329,2.266131,2.625857,3.875073
3,Random Forest,0.970440,0.970292,0.958602,5.595621,5.609549,6.621878,2.053292,2.189217,2.804643
4,SVR-RBF,0.910667,0.905969,0.884678,9.727480,9.979966,11.052224,4.837621,5.025854,6.095166
5,Polynomial-D2,0.856054,0.834930,0.769223,12.347925,13.222925,15.634716,7.309489,8.978690,10.936291
6,Linear Regression,0.648494,0.555005,0.468333,19.295675,21.710561,23.730863,14.611895,15.896853,17.529435


In [13]:
#BLOCK 3 — GPR - RQ
gpr_comparison_results = {}
for model_name, factory, filename in [
    ("GPR-RationalQuadratic", create_gpr_rq, "GPR_RQ_Feature_Comparison.xlsx"),
]:
    results = []
    for feature_name, features in feature_sets.items():
        print(feature_name, model_name)
        gpr = factory()
        gpr.fit(X_train_all[features], y_train)
        prediction = gpr.predict(X_test_all[features])
        results.append({"Feature Set": feature_name, "Features": len(features),
                        "Model": model_name, **metrics(y_test, prediction)})
        del gpr
    results_df = pd.DataFrame(results)
    gpr_comparison_results[model_name] = results_df
    display(results_df)
    results_df.to_excel(filename, index=False)


Pearson-29 GPR-RationalQuadratic


C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Consensus-15 GPR-RationalQuadratic


C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Strict-6 GPR-RationalQuadratic


,Feature Set,Features,Model,R2,RMSE,MAE
0,Pearson-29,29,GPR-RationalQuadratic,0.976322,5.008051,1.737945
1,Consensus-15,15,GPR-RationalQuadratic,0.974456,5.201564,1.802174
2,Strict-6,6,GPR-RationalQuadratic,0.964360,6.144135,2.540497


In [14]:
#BLOCK 4 — GPR - RBF
def rbf_gpr(ls=1.0, ls_bounds=(1e-2, 1e2), noise=1e-2, noise_bounds=(1e-6, 1), restarts=0):
    kernel = ConstantKernel(1.0) * RBF(ls, ls_bounds) + WhiteKernel(noise, noise_bounds)
    return Pipeline([('scale', StandardScaler()),
                     ('gpr', GaussianProcessRegressor(kernel, normalize_y=True,
                                                      n_restarts_optimizer=restarts,
                                                      random_state=42))])

# Optimized length scales
best_ls = {"Pearson-29": 2.5,"Consensus-15": 1.5,"Strict-6": 0.4}

final = []

for name, feats in feature_sets.items():
    m = rbf_gpr(
        ls=best_ls[name],
        ls_bounds="fixed",
        restarts=2
    ).fit(X_train_all[feats], y_train)

    print(f"\n{name}")
    print("Selected length scale:", best_ls[name])
    print("Optimized kernel:", m.named_steps["gpr"].kernel_)

    final.append({
        "Feature Set": name,
        "Features": len(feats),
        "Model": "GPR-RBF",
        **metrics(y_test, m.predict(X_test_all[feats]))
    })

final_results = pd.DataFrame(final)

display(final_results)


Pearson-29
Selected length scale: 2.5
Optimized kernel: 1.13**2 * RBF(length_scale=2.5) + WhiteKernel(noise_level=0.0409)

Consensus-15
Selected length scale: 1.5
Optimized kernel: 0.945**2 * RBF(length_scale=1.5) + WhiteKernel(noise_level=0.0446)

Strict-6
Selected length scale: 0.4
Optimized kernel: 0.683**2 * RBF(length_scale=0.4) + WhiteKernel(noise_level=0.0508)


,Feature Set,Features,Model,R2,RMSE,MAE
0,Pearson-29,29,GPR-RBF,0.951396,7.175141,3.818936
1,Consensus-15,15,GPR-RBF,0.948186,7.408275,3.937398
2,Strict-6,6,GPR-RBF,0.934085,8.355746,4.621465


In [15]:
# Blind predictions: all non-GPR models
development, blind = load_blind_data()
all_predictions, performance, nn_predictions, nn_performance = [], [], [], []
for feature_name, features in feature_sets.items():
    Xtr, Xte = development[features], blind[features]
    ytr, y_true = development[target], blind[target]
    prediction_table = pd.DataFrame({"Material": blind["Material"], "Actual_Tc": y_true})
    models = create_models()
    for model_name, model in models.items():
        print(feature_name, model_name)
        model.fit(Xtr, ytr)
        pred = model.predict(Xte)
        prediction_table[model_name] = pred
        performance.append({"Feature_Set": feature_name, "Model": model_name,
                            **metrics(y_true, pred)})
    prediction_table["Mean_Prediction"] = prediction_table[list(models)].mean(axis=1)
    prediction_table["Prediction_SD"] = prediction_table[list(models)].std(axis=1)
    prediction_table["Feature_Set"] = feature_name
    all_predictions.append(prediction_table)

    nn_mean, nn_sd = nn_ensemble(Xtr, ytr, Xte, NN_BLIND_SEEDS)
    nn_performance.append({"Feature_Set": feature_name, "Number_of_Features": len(features),
                           **metrics(y_true, nn_mean)})
    nn_predictions.append(pd.DataFrame({"Material": blind["Material"], "Actual_Tc": y_true,
        "NN_Prediction": nn_mean, "NN_Ensemble_SD": nn_sd, "Feature_Set": feature_name}))

performance_df, prediction_df = pd.DataFrame(performance), pd.concat(all_predictions, ignore_index=True)
nn_performance_df = pd.DataFrame(nn_performance)
nn_prediction_df = pd.concat(nn_predictions, ignore_index=True)
blind_all_performance = pd.concat([
    performance_df, nn_performance_df.assign(Model="Deep Ensemble NN")], ignore_index=True)
blind_all_predictions = prediction_df.merge(
    nn_prediction_df[["Feature_Set", "Material", "NN_Prediction", "NN_Ensemble_SD"]],
    on=["Feature_Set", "Material"], validate="one_to_one")
display(blind_all_performance)
display(blind_all_predictions)


for names, classical_file, nn_file in [
    (["Consensus-15", "Strict-6"], "Blind10_Consensus15_vs_Strict6.xlsx",
     "Blind10_NN_Consensus15_vs_Strict6.xlsx"),
    (["Pearson-29"], "Blind10_F29_Full_Features.xlsx", "Blind10_NN_FullF29.xlsx"),
]:
    save_blind(classical_file,
        performance_df[performance_df.Feature_Set.isin(names)],
        prediction_df[prediction_df.Feature_Set.isin(names)])
    save_blind(nn_file,
        nn_performance_df[nn_performance_df.Feature_Set.isin(names)],
        nn_prediction_df[nn_prediction_df.Feature_Set.isin(names)])


Development: (13494, 31) Blind: (10, 32)
Pearson-29 Linear
Pearson-29 Polynomial-D2
Pearson-29 SVR-RBF
Pearson-29 Random Forest
Pearson-29 XGBoost
Pearson-29 LightGBM
Consensus-15 Linear
Consensus-15 Polynomial-D2
Consensus-15 SVR-RBF
Consensus-15 Random Forest
Consensus-15 XGBoost
Consensus-15 LightGBM
Strict-6 Linear
Strict-6 Polynomial-D2
Strict-6 SVR-RBF
Strict-6 Random Forest
Strict-6 XGBoost
Strict-6 LightGBM


,Feature_Set,Model,R2,RMSE,MAE,Number_of_Features
0,Pearson-29,Linear,0.748878,12.153447,9.339915,NaN
1,Pearson-29,Polynomial-D2,0.468513,17.680854,8.731597,NaN
2,Pearson-29,SVR-RBF,0.296167,20.346582,8.015693,NaN
3,Pearson-29,Random Forest,0.999326,0.629433,0.484504,NaN
4,Pearson-29,XGBoost,0.998238,1.017885,0.808752,NaN
5,Pearson-29,LightGBM,0.998962,0.781546,0.675228,NaN
6,Consensus-15,Linear,0.628326,14.785554,12.997559,NaN
7,Consensus-15,Polynomial-D2,0.449658,17.991737,9.824345,NaN
8,Consensus-15,SVR-RBF,0.222623,21.383202,8.307925,NaN
9,Consensus-15,Random Forest,0.997789,1.140490,0.705449,NaN


,Material,Actual_Tc,Linear,Polynomial-D2,SVR-RBF,Random Forest,XGBoost,LightGBM,Mean_Prediction,Prediction_SD,Feature_Set,NN_Prediction,NN_Ensemble_SD
0,Yb0.95Pr0.05Ba1.7Sr0.3Cu3O6.91,81.000,65.333633,84.395914,82.214674,80.999410,80.738068,80.790665,79.078727,6.876153,Pearson-29,80.520358,0.720325
1,Hf0.99Rh0.01,0.850,-0.694367,-6.601690,1.750226,2.383760,1.401488,1.190081,-0.095084,3.350709,Pearson-29,1.693834,0.767451
2,Bi0.5Cd0.125Pb0.25Sn0.125,8.200,22.076262,12.995687,11.187454,7.807117,7.492350,7.496878,11.509291,5.655355,Pearson-29,8.804211,0.686890
3,La1Pt0.3Ni0.7Si1,1.655,-2.924800,3.306319,1.772044,2.058405,2.794417,2.434793,1.573530,2.269113,Pearson-29,2.860371,0.950024
4,Sr0.9Sm0.1Cu1O2,44.000,63.636015,98.598817,107.997012,43.648000,45.633709,43.232828,67.124397,29.179364,Pearson-29,46.456116,1.743519
5,Eu1.3Pr0.2Ce0.5Sr2Cu2Nb1O10,13.400,10.687768,14.712646,12.546035,14.100860,13.033423,13.419596,13.083388,1.402356,Pearson-29,13.212346,1.097086
6,La1.95Bi0.05Cu0.99Zn0.01O4.093,22.700,44.830178,19.680181,18.544661,22.216113,22.236435,21.885098,24.898778,9.881663,Pearson-29,20.667797,1.650814
7,Ba1Ni2As1.846P0.154,3.280,14.742894,7.498169,6.268862,3.602663,3.398228,3.842738,6.558926,4.337069,Pearson-29,2.469306,0.963197
8,Zr25.8V51.3Nb22.9,4.340,5.379589,7.953460,6.742285,4.916970,6.507453,5.543158,6.173819,1.114432,Pearson-29,7.415785,1.094439
9,B1C1Mo2,6.325,5.573549,9.583455,5.784930,6.403020,5.647497,4.972614,6.327511,1.659391,Pearson-29,6.725428,0.724615


In [22]:
# Blind GPR-RQ
development, blind = load_blind_data()
Xtr, _, ytr, _ = train_test_split(development[full_features], development[target],
                                 test_size=TEST_SIZE, random_state=RANDOM_STATE)
results = pd.DataFrame({"Material": blind["Material"], "Actual_Tc": blind[target]})
gpr_blind_performance = []
column_names = {"Pearson-29": "GPR_RQ_Full", "Consensus-15": "GPR_RQ_F15", "Strict-6": "GPR_RQ_F6"}
for feature_name, features in feature_sets.items():
    print("Training blind GPR-RQ:", feature_name)
    model = create_blind_gpr_rq()
    model.fit(Xtr[features], ytr)
    joblib.dump(model, GPR_MODEL_FILES[feature_name])
    prediction = model.predict(blind[features])
    results[column_names[feature_name]] = prediction
    gpr_blind_performance.append({"Feature_Set": feature_name, "Model": "GPR-RQ",
                                  **metrics(blind[target], prediction)})
    del model

display(results)
display(pd.DataFrame(gpr_blind_performance))
results[["Material", "Actual_Tc", "GPR_RQ_F15", "GPR_RQ_F6"]].to_excel(
    "GPR_RQ_blind_predictions.xlsx", index=False)
results[["Material", "Actual_Tc", "GPR_RQ_Full"]].to_excel(
    "GPR_RQ_blind_predictions_full.xlsx", index=False)
    

Development: (13494, 31) Blind: (10, 31)
Training blind GPR-RQ: Pearson-29


C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Training blind GPR-RQ: Consensus-15


C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Training blind GPR-RQ: Strict-6


,Material,Actual_Tc,GPR_RQ_Full,GPR_RQ_F15,GPR_RQ_F6
0,Yb0.95Pr0.05Ba1.7Sr0.3Cu3O6.91,81.000,80.533024,81.286995,81.889521
1,Hf0.99Rh0.01,0.850,1.346859,1.826312,2.358535
2,Bi0.5Cd0.125Pb0.25Sn0.125,8.200,8.395811,5.899540,7.194748
3,La1Pt0.3Ni0.7Si1,1.655,2.504051,2.729267,3.487405
4,Sr0.9Sm0.1Cu1O2,44.000,44.530638,44.105993,44.489833
5,Eu1.3Pr0.2Ce0.5Sr2Cu2Nb1O10,13.400,12.990806,13.036893,12.105848
6,La1.95Bi0.05Cu0.99Zn0.01O4.093,22.700,22.231406,21.930023,22.377856
7,Ba1Ni2As1.846P0.154,3.280,3.286514,3.298278,3.378414
8,Zr25.8V51.3Nb22.9,4.340,6.428694,6.498363,6.275533
9,B1C1Mo2,6.325,6.341865,6.385596,6.565806


,Feature_Set,Model,R2,RMSE,MAE
0,Pearson-29,GPR-RQ,0.998936,0.790932,0.552920
1,Consensus-15,GPR-RQ,0.997810,1.134910,0.811435
2,Strict-6,GPR-RQ,0.997744,1.151854,0.961659


In [20]:
print(feature_sets.keys())
print(GPR_MODEL_FILES)

dict_keys(['Pearson-29', 'Consensus-15', 'Strict-6'])
{'Pearson-29': 'GPR_RQ_Full-F29.pkl', 'Consensus-15': 'GPR_RQ_Consensus15.pkl', 'Strict-6': 'GPR_RQ_Strict6.pkl'}
